# L4 讲义：线性回归（Linear Regression）、普通最小二乘（Ordinary Least Squares, OLS）与岭回归（Ridge Regression）

本笔记本是一份可独立学习的 L4 幻灯片替代材料。它系统展开课程中的全部主题，补全动画中逐步呈现的证明，解答幻灯片练习，并补充两本配套教材中最相关的结果。

**主要资料来源**

- *L4: Linear regression, OLS, ridge regression*，幻灯片第 3–37 页（由于 PDF 含动画帧，对应 PDF 实际第 4–176 页）。
- Francis Bach，*Learning Theory from First Principles*，第 3 章，书本第 45–66 页（PDF 实际第 61–82 页），并选取第 12.1–12.2 节和第 14.1 节中的拓展内容。
- Shalev-Shwartz 与 Ben-David，*Understanding Machine Learning: Solution Manual*，PDF/书本第 26 页及第 64–65 页：绝对损失回归的线性规划形式、秩判据、带截距时的中心化，以及标量软阈值。

**阅读指南**

第 1–7 节建立回归问题并讲解一维示例；第 8–15 节推导并分析普通最小二乘（ordinary least squares, OLS）；第 16–22 节介绍岭回归（ridge regression）与一般正则化；第 23–24 节给出有助于理解高维和随机设计行为的进阶拓展；第 25–26 节汇总例题详解和最终检查清单。

**公式与术语约定。** 核心专业术语首次出现时采用 **中文**（English）的形式标注。行内数学使用 `$...$`；独立公式使用 `$$...$$`。全文中，$N$ 表示样本量，$d$ 表示拟合参数的个数；若截距被表示为一个特征，则截距也计入参数总数。教材中常用 $n$ 表示本笔记中的 $N$。


## 1. 回归设定（Regression Setup）与记号（Notation）

一个**监督学习样本**（Supervised Sample）为

$$
S=\{(x_1,y_1),\ldots,(x_N,y_N)\},
\qquad x_i\in\mathbb{R}^p,\quad y_i\in\mathbb{R}.
$$

**分类**（Classification）预测类别，例如 $y\in\{0,1\}$；**回归**（Regression）预测数值响应。我们的目标是寻找一个规则 $h:\mathbb{R}^p\to\mathbb{R}$，用它预测新输入的响应。

**特征映射**（Feature Map）$\phi:\mathbb{R}^p\to\mathbb{R}^d$ 生成模型所用的特征向量。**参数线性**（Linear in Parameters）的预测器具有形式

$$
f_w(x)=\phi(x)^\top w,
\qquad w\in\mathbb{R}^d.
$$

将特征向量排列成**设计矩阵**（Design Matrix），并将响应排列成向量：

$$
X=
\begin{bmatrix}
\phi(x_1)^\top\\
\vdots\\
\phi(x_N)^\top
\end{bmatrix}\in\mathbb{R}^{N\times d},
\qquad
y=
\begin{bmatrix}y_1\\ \vdots\\ y_N\end{bmatrix}\in\mathbb{R}^N.
$$

于是拟合响应向量为 $Xw$。**仿射模型**（Affine Model）$w_0+x^\top\beta$ 可通过令 $\phi(x)=(1,x^\top)^\top$ 且 $w=(w_0,\beta^\top)^\top$ 纳入这一框架。除非另有说明，本文均假定以这种方式加入**截距**（Intercept）。

未中心化的**经验特征协方差**（Empirical Feature Covariance）为

$$
\widehat\Sigma=\frac{1}{N}X^\top X.
$$

不要混淆输入维数 $p$、特征维数 $d$ 与样本量 $N$。

*资料来源：L4 幻灯片 3–4；Bach，第 3.1–3.2 节。*


## 2. 经验风险最小化（Empirical Risk Minimization, ERM）与损失函数（Loss Functions）的选择

给定损失函数 $\ell(\widehat y,y)$ 和假设类 $\mathcal H$，**经验风险最小化**（Empirical Risk Minimization, ERM）求解

$$
\widehat h\in\arg\min_{h\in\mathcal H}
\widehat R(h),
\qquad
\widehat R(h)=\frac1N\sum_{i=1}^N\ell(h(x_i),y_i).
$$

**平方损失**（Squared Loss）给出**最小二乘回归**（Least-Squares Regression）：

$$
\ell(\widehat y,y)=(\widehat y-y)^2,
\qquad
\widehat w\in\arg\min_w\frac1N\|Xw-y\|_2^2.
$$

它是凸且可微的，会对大残差施加较重惩罚；对于参数线性模型，它有闭式解；在高斯噪声下，它等价于**最大似然估计**（Maximum-Likelihood Estimation, MLE）；在总体层面，它以**条件均值**（Conditional Mean）为目标。

**绝对损失**（Absolute Loss）对大残差更稳健，并以**条件中位数**（Conditional Median）为目标：

$$
\min_w\frac1N\sum_{i=1}^N|y_i-\phi(x_i)^\top w|.
$$

引入 $s_i\ge 0$ 后，它可写成**线性规划**（Linear Program, LP）：

$$
\begin{aligned}
\min_{w,s}\quad &\sum_{i=1}^N s_i\\
\text{subject to}\quad
&\phi(x_i)^\top w-y_i\le s_i,\\
&y_i-\phi(x_i)^\top w\le s_i,
\qquad i=1,\ldots,N.
\end{aligned}
$$

对于 $\tau\in(0,1)$，**检验损失**（Check Loss，也称 Pinball Loss）为

$$
\rho_\tau(u)=u\bigl(\tau-\mathbf 1\{u<0\}\bigr).
$$

最小化 $\sum_i\rho_\tau(y_i-h(x_i))$ 即进行**分位数回归**（Quantile Regression），并以条件 $\tau$ 分位数为目标。幻灯片将其称为形似泄漏 ReLU 的损失；在统计学中，check loss 或 pinball loss 是标准名称。不同损失函数回答不同的统计问题，因此平方损失是一种建模选择，而不是普遍适用的默认选项。

*资料来源：L4 幻灯片 4；Bach，第 2.2.3 节与第 3.2 节；解答手册，第 9 章习题 1 解答（第 26 页）。*


## 3. 一元线性回归（Simple Linear Regression）：完整推导

对于标量输入，考虑

$$
\widehat y_i=w_0+wx_i,
\qquad
L(w_0,w)=\sum_{i=1}^N(y_i-w_0-wx_i)^2.
$$

**一阶条件**（First-Order Conditions）为

$$
\frac{\partial L}{\partial w_0}
=-2\sum_{i=1}^N(y_i-w_0-wx_i)=0,
$$

$$
\frac{\partial L}{\partial w}
=-2\sum_{i=1}^N x_i(y_i-w_0-wx_i)=0.
$$

定义 $\bar x=N^{-1}\sum_i x_i$ 和 $\bar y=N^{-1}\sum_i y_i$。第一个方程给出

$$
\widehat w_0=\bar y-\widehat w\bar x.
$$

将其代入第二个方程后得到

$$
\widehat w
=\frac{\sum_{i=1}^N(x_i-\bar x)(y_i-\bar y)}
{\sum_{i=1}^N(x_i-\bar x)^2}
=\frac{\widehat{\operatorname{Cov}}(X,Y)}
{\widehat{\operatorname{Var}}(X)}.
$$

若 $s_x$、$s_y$ 与经验协方差采用相同的分母，且 $r$ 为**皮尔逊相关系数**（Pearson Correlation Coefficient），则

$$
\widehat w=r\frac{s_y}{s_x},
\qquad
\widehat y-\bar y=r\frac{s_y}{s_x}(x-\bar x).
$$

由此可得：

- 拟合直线经过 $(\bar x,\bar y)$；
- 同时对两个变量进行**中心化**（Centering）会消去截距，但不改变斜率；
- 改变度量单位会改变斜率，但不会改变相关系数；
- 如果所有 $x_i$ 都相同，则斜率没有定义。

中心化恒等式是精确的：

$$
\min_{w_0,w}\sum_i(y_i-w_0-wx_i)^2
=\min_w\sum_i\bigl[(y_i-\bar y)-w(x_i-\bar x)\bigr]^2.
$$

*资料来源：L4 幻灯片 6–7；解答手册，第 25 章习题 1 解答（第 64 页）。*


## 4. 皮尔逊父子身高示例、均值回归（Regression Toward the Mean）与均方根误差（Root Mean Squared Error, RMSE）

课程采用以下近似数值：

$$
\bar x=68,\quad s_x=2.7,
\qquad
\bar y=69,\quad s_y=2.7,
\qquad r=0.5,
$$

其中 $x$ 是父亲的身高，$y$ 是儿子的身高，单位均为英寸。因此

$$
\widehat w=r\frac{s_y}{s_x}=0.5,
\qquad
\widehat w_0=\bar y-\widehat w\bar x=35,
$$

所以 $\widehat y=35+0.5x$。对于身高为 $64$ 英寸的父亲，

$$
\widehat y=35+0.5(64)=67.
$$

预测值不是 $65$。该父亲的身高比均值低 $(64-68)/2.7$ 个标准差，而预测的儿子身高仅比其均值低上述标准差数的 $r$ 倍。当 $|r|<1$ 时，预测响应的标准化值更接近零；这就是**均值回归**（Regression Toward the Mean）。它并不是说个体会在生理上向均值变化。

下列恒等式采用 $1/N$ 约定：

$$
s_x^2=\frac1N\sum_{i=1}^N(x_i-\bar x)^2,\qquad
s_y^2=\frac1N\sum_{i=1}^N(y_i-\bar y)^2,\qquad
r=\frac{N^{-1}\sum_{i=1}^N(x_i-\bar x)(y_i-\bar y)}{s_xs_y}.
$$

在此约定下，对于带截距的最小二乘，**经验残差均方**（Empirical Residual Mean Square）满足

$$
\frac1N\sum_{i=1}^N(y_i-\widehat w_0-\widehat wx_i)^2
=s_y^2(1-r^2).
$$

因此

$$
\operatorname{RMSE}=s_y\sqrt{1-r^2}.
$$

为了验证这一点，先中心化 $x$ 和 $y$，再代入 $\widehat w=r s_y/s_x$：

$$
\begin{aligned}
\operatorname{MSE}
&=\frac1N\sum_i\bigl[(y_i-\bar y)-\widehat w(x_i-\bar x)\bigr]^2\\
&=s_y^2-2\widehat w\,r s_xs_y+\widehat w^2s_x^2\\
&=s_y^2(1-r^2).
\end{aligned}
$$

若 $s_y^2$ 改用分母为 $N-1$ 的常规样本方差，则同一残差 MSE 为 $\frac{N-1}{N}s_y^2(1-r^2)$。该恒等式是一元 OLS 的样本内恒等式。新数据上的测试 RMSE 不一定等于这一数值。

*资料来源：L4 幻灯片 8。*


## 5. 为什么相关系数（Correlation Coefficient）和汇总统计量（Summary Statistics）并不足够

**皮尔逊相关系数**（Pearson Correlation Coefficient）衡量**线性关联**（Linear Association），而非**因果关系**（Causality）。较大的 $|r|$ 可能源于**混杂因素**（Confounder）、共同的时间趋势、选择效应、受限的测量范围，或某个具有强影响力的点。反之，强烈的非线性关系也可能具有 $r\approx0$。

**安斯库姆四重奏**（Anscombe's Quartet）包含若干数据集：它们的样本均值、方差、相关系数和 OLS 直线几乎相同，但几何形态显著不同——一个近似线性，一个呈曲线，一个由**离群点**（Outlier）主导，另一个包含**高杠杆点**（High-Leverage Point）。幻灯片还展示了可以人为构造具有相同汇总统计量的任意点云。

因此，回归分析至少应检查：

- 响应与每个重要特征之间的散点图；
- 残差 $e_i=y_i-\widehat y_i$ 对拟合值的图；
- 极端残差和高杠杆输入；
- 残差散布是否变化；若变化，可能表明存在**异方差性**（Heteroscedasticity）；
- 非线性模式与子群结构；
- 是否在观测到的特征范围之外进行**外推**（Extrapolation）。

OLS 概括的是**条件关联**（Conditional Association）。它本身既不能识别干预效应，也不足以支持因果结论。

*资料来源：L4 幻灯片 9–10。*


## 6. 总体回归（Population Regression）：为什么条件均值（Conditional Mean）最优

可测预测器 $h$ 的**总体平方损失风险**（Population Squared-Loss Risk）为

$$
R(h)=\mathbb E\bigl[(Y-h(X))^2\bigr].
$$

令

$$
\eta(x)=\mathbb E[Y\mid X=x].
$$

对任意固定的 $x$ 和任意取值 $z$，

$$
\begin{aligned}
\mathbb E[(Y-z)^2\mid X=x]
&=\mathbb E[(Y-\eta(x)+\eta(x)-z)^2\mid X=x]\\
&=\operatorname{Var}(Y\mid X=x)+(z-\eta(x))^2,
\end{aligned}
$$

因为 $\mathbb E[Y-\eta(x)\mid X=x]=0$。第一项与 $z$ 无关，第二项非负。因此

$$
h^*(x)=\eta(x)=\mathbb E[Y\mid X=x]
$$

对每个 $x$ 都最小化**条件风险**（Conditional Risk），因而也最小化 $R(h)$。更一般地，

$$
R(h)=R(h^*)+\mathbb E\bigl[(h(X)-h^*(X))^2\bigr],
$$

其中**不可约贝叶斯风险**（Irreducible Bayes Risk）为

$$
R(h^*)=\mathbb E\bigl[\operatorname{Var}(Y\mid X)\bigr].
$$

从几何上看，**条件期望**（Conditional Expectation）是 $Y$ 到闭 $L^2$ 子空间（由 $X$ 的函数构成）上的**正交投影**（Orthogonal Projection）；上述风险恒等式就是相应的勾股定理。

$(X,Y)$ 的联合分布未知，因此通常无法直接计算 $\eta(x)$。学习过程将 $h$ 限制在一个可处理的函数类中，并以**经验风险**（Empirical Risk）近似**总体风险**（Population Risk）。如果 $\eta$ 不在该函数类中，剩余差距就是**近似误差**（Approximation Error）或**模型误设**（Model Misspecification）。

*资料来源：L4 幻灯片 11–13；Bach，第 2.2.3 节与第 3.2 节。*


## 7. 判别式回归（Discriminative Regression）、非线性特征（Nonlinear Features）与模型误设（Model Misspecification）

**判别式回归模型**（Discriminative Regression Model）直接描述 $Y$ 在给定 $X$ 时的分布，而不是对完整**联合分布**（Joint Distribution）$\mathbb P(X,Y)$ 建模。**线性特征模型**（Linear Feature Model）假设

$$
\mathbb E[Y\mid X=x]\approx f_w(x)=\phi(x)^\top w.
$$

“线性”指的是关于 $w$ 线性，而不一定是关于 $x$ 线性。例如，**多项式回归**（Polynomial Regression）可以采用

$$
\phi(x)=(1,x,x^2,\ldots,x^k)^\top
$$

此外还可以使用**样条**（Spline）、**交互项**（Interaction Term）、**周期特征**（Periodic Feature），或在其他任务中学得的固定表示。所得曲线关于 $x$ 可以高度非线性，而 OLS 关于 $w$ 仍然是二次优化问题。

学习问题可以分解为三个选择：

1. 特征映射 $\phi$，它决定可用的函数集合；
2. 损失函数，它决定总体层面的目标；
3. 算法或**正则化器**（Regularizer），它在参数之间作出选择并控制稳定性。

更大的特征空间会降低近似误差，却可能增加**估计方差**（Estimation Variance）和**过拟合**（Overfitting）。当嵌套的特征类扩大时，训练误差只会下降，但测试误差未必如此。这是**偏差—方差权衡**（Bias–Variance Trade-Off）的首次出现。

用于 $0/1$ 分类的二元 VC 维不能直接分析无界平方损失。适用于实值函数的容量工具包括**伪维数**（Pseudo-Dimension）和 **Rademacher 复杂度**（Rademacher Complexity）。幻灯片提到这一问题，但将一般理论留待后续；本讲改为在线性固定设计模型下给出显式的有限样本计算。

*资料来源：L4 幻灯片 15–17；Bach，第 2.3.2 节与第 3.2 节。*


## 8. 多元普通最小二乘（Multivariate OLS）与正规方程（Normal Equations）

**普通最小二乘**（Ordinary Least Squares, OLS）最小化

$$
\widehat R(w)=\frac1N\|y-Xw\|_2^2.
$$

展开得

$$
\widehat R(w)=\frac1N
\left(y^\top y-2w^\top X^\top y+w^\top X^\top Xw\right).
$$

因此

$$
\nabla\widehat R(w)=\frac2N(X^\top Xw-X^\top y),
\qquad
\nabla^2\widehat R(w)=\frac2N X^\top X=2\widehat\Sigma.
$$

令梯度为零，得到**正规方程**（Normal Equations）

$$
X^\top X\widehat w=X^\top y.
$$

如果 $X$ 具有**满列秩**（Full Column Rank）$d$（因此必有 $d\le N$），则 $X^\top X$ 为**正定矩阵**（Positive-Definite Matrix），极小解唯一：

$$
\widehat w_{\mathrm{OLS}}=(X^\top X)^{-1}X^\top y.
$$

**秩判据**（Rank Criterion）为

$$
X^\top X\text{ is invertible}
\quad\Longleftrightarrow\quad
\operatorname{rank}(X)=d
\quad\Longleftrightarrow\quad
\text{the columns of }X\text{ are linearly independent}.
$$

事实上，$u^\top X^\top Xu=\|Xu\|_2^2$，因此 $\operatorname{null}(X^\top X)=\operatorname{null}(X)$。

**完全多重共线性**（Perfect Multicollinearity）会违反此条件。**近似共线性**（Near Collinearity）不会破坏唯一性，却会使估计在数值和统计意义上都不稳定。

*资料来源：L4 幻灯片 16；Bach，第 3.2–3.3.1 节；解答手册，第 9 章习题 2 解答（第 26 页）。*


## 9. 投影几何（Projection Geometry）、帽子矩阵（Hat Matrix）与稳定计算（Stable Computation）

定义**帽子矩阵**（Hat Matrix）

$$
H=X(X^\top X)^{-1}X^\top.
$$

当设计矩阵满列秩时，$H^\top=H$ 且 $H^2=H$，所以 $H$ 是到列空间 $\operatorname{col}(X)$ 上的**正交投影算子**（Orthogonal Projector）。**拟合响应**（Fitted Response）和**残差**（Residual）分别为

$$
\widehat y=Hy,
\qquad
e=y-\widehat y=(I-H)y.
$$

正规方程等价于

$$
X^\top e=0,
$$

因此残差与每个拟合特征方向都正交。对角元 $h_{ii}$ 是观测 $i$ 的**杠杆值**（Leverage）；异常大的杠杆值意味着该观测的特征向量能够强烈影响它自身的拟合结果。

逆矩阵公式适合代数推导，但数值程序不应显式构造 $(X^\top X)^{-1}$。构造 $X^\top X$ 会使**谱条件数**（Spectral Condition Number）平方：

$$
\kappa_2(X^\top X)=\kappa_2(X)^2.
$$

推荐的方法包括：

- **QR 分解**（QR Factorization）$X=QR$，随后求解 $R\widehat w=Q^\top y$；
- **奇异值分解**（Singular Value Decomposition, SVD）或最小二乘例程，尤其适用于接近**秩亏**（Rank Deficiency）的情形；
- 对于超大规模问题，使用 LSQR 或 LSMR；这类方法只需 $X$ 与 $X^\top$ 的矩阵—向量乘法。

**共轭梯度**（Conjugate Gradient, CG）只适用于适当的**对称正定**（Symmetric Positive Definite, SPD）系统，例如岭回归方程组，而且通常需要**预条件**（Preconditioning）；直接将其用于 $X^\top X$ 会继承平方后的条件数。

在 Python/NumPy 中，概念上的推荐写法是 `numpy.linalg.lstsq(X, y)`，而不是 `inv(X.T @ X) @ X.T @ y`。

*资料来源：Bach，第 3.3.2–3.3.3 节；投影细节是对 OLS 动画幻灯片的补充。*


## 10. 固定设计（Fixed Design）、随机设计（Random Design）与观测模型（Observation Model）

分析一个估计量时，必须明确哪些量是随机的。

**固定设计**（Fixed Design）：将 $X$ 的各行视为确定量，只有响应发生变化。性能在相同输入点上评估，但可以使用重新采样的响应。这是一个样本内去噪问题。

**随机设计**（Random Design）：训练输入是随机的，性能在从总体中抽取的新输入上评估。这是通常意义下的泛化问题。

课程首先采用固定设计且模型设定正确的线性模型

$$
y=Xw_*+\varepsilon,
\qquad
\mathbb E[\varepsilon]=0,
\qquad
\operatorname{Cov}(\varepsilon)=\sigma^2I_N.
$$

这些假设的含义是：

- **设定正确**（Well-Specified）：设计点处的条件均值等于 $Xw_*$；
- **零均值**（Zero Mean）：误差不会系统性地平移响应；
- **同方差**（Homoscedasticity）：每个响应的方差均为 $\sigma^2$；
- **两两不相关**（Pairwise Uncorrelated）：噪声协方差的非对角元素为零。

条件 $\operatorname{Cov}(\varepsilon)=\sigma^2I_N$ 只说明同方差与两两不相关，并不能推出独立。**独立性**（Independence）是额外假设；在后文的 iid 高斯模型中，独立性成立。

下文关于 OLS 均值、协方差和期望风险的计算只需要上述一阶、二阶矩条件；固定设计测试风险还要求新的测试噪声与训练噪声独立。对这些计算而言，高斯性是更强的假设；它只在精确似然表述和某些精确分布结果中才是必需的。在模型误设下，会保留一个近似误差项；在异方差或相关噪声下，若噪声协方差为 $C$，协方差公式应以 $C$ 代替 $\sigma^2I$。

*资料来源：L4 幻灯片 18；Bach，第 3.4–3.5 节。*


## 11. 高斯似然（Gaussian Likelihood）与两种噪声方差估计量（Noise-Variance Estimators）

如果固定设计下的噪声服从**高斯分布**（Gaussian Distribution），

$$
\varepsilon_i\overset{\mathrm{iid}}\sim\mathcal N(0,\sigma^2),
$$

则

$$
p(y\mid w,\sigma^2)
=(2\pi\sigma^2)^{-N/2}
\exp\left(-\frac{\|y-Xw\|_2^2}{2\sigma^2}\right).
$$

当 $\sigma^2$ 固定时，最大化似然等价于最小化**残差平方和**（Residual Sum of Squares, RSS），因此 $w$ 的最大似然估计就是 OLS。如果最小残差平方和为正，则对 $w$ 和 $\sigma^2$ 联合最大化可得

$$
\widetilde\sigma^2_{\mathrm{MLE}}
=\frac1N\|y-X\widehat w\|_2^2.
$$

该**最大似然估计**（Maximum-Likelihood Estimate, MLE）向下有偏，因为拟合的 $d$ 个参数已经消耗了一部分噪声自由度。当设计满列秩且 $N>d$ 时，**无偏估计量**（Unbiased Estimator）为

$$
\widehat\sigma^2
=\frac{\|y-X\widehat w\|_2^2}{N-d}.
$$

这两个公式回答不同的问题：除以 $N$ 来自最大似然；除以 $N-d$ 则是为了校正期望。当设计满列秩且 $N>d$ 时，高斯噪声使最小 RSS 几乎必然为正。若 $N=d$，OLS 会精确拟合，RSS 为零，$\mathrm{RSS}/(N-d)$ 没有定义；当 $\sigma^2\downarrow0$ 时似然无界，因此在参数域 $\sigma^2>0$ 内不存在联合 MLE。当有效秩不等于 $d$，或方差模型不是同方差模型时，两式都不能原样使用。

*资料来源：L4 幻灯片 18（MLE 说明）；Bach，第 3.5 节及习题 3.1–3.2。*


## 12. 固定设计风险（Fixed-Design Risk）及其偏差—方差分解（Bias–Variance Decomposition）

令相同设计点处的一个新响应为

$$
y'=Xw_*+\varepsilon',
\qquad
\varepsilon'\text{ independent of the training noise}.
$$

定义**固定设计测试风险**（Fixed-Design Test Risk）

$$
R_X(w)=\mathbb E_{\varepsilon'}\left[\frac1N\|y'-Xw\|_2^2\right].
$$

由于交叉项的期望为零，

$$
R_X(w)=\sigma^2+\frac1N\|X(w-w_*)\|_2^2
=\sigma^2+\|w-w_*\|_{\widehat\Sigma}^2,
$$

其中 $\|v\|_{\widehat\Sigma}^2=v^\top\widehat\Sigma v$。因此 $R_X^*=\sigma^2$，且

$$
R_X(w)-R_X^*=\|w-w_*\|_{\widehat\Sigma}^2.
$$

对于随机估计量 $\widetilde w$，加上再减去 $\mathbb E[\widetilde w]$：

$$
\begin{aligned}
\mathbb E[R_X(\widetilde w)]-R_X^*
&=\|\mathbb E[\widetilde w]-w_*\|_{\widehat\Sigma}^2\\
&\quad+\mathbb E\|\widetilde w-\mathbb E[\widetilde w]\|_{\widehat\Sigma}^2.
\end{aligned}
$$

第一项是**偏差平方**（Squared Bias），第二项是**方差**（Variance）；二者都在 $X$ 所诱导的**预测几何**（Prediction Geometry）中度量。当目标是预测而不是**参数恢复**（Parameter Recovery）时，这种度量比欧氏参数误差更相关。

*资料来源：Bach，命题 3.3。*


## 13. OLS 的无偏性（Unbiasedness）、协方差（Covariance）与精确预测误差（Exact Prediction Error）

在固定设计模型和满列秩条件下，

$$
\widehat w
=(X^\top X)^{-1}X^\top(Xw_*+\varepsilon)
=w_*+(X^\top X)^{-1}X^\top\varepsilon.
$$

因此

$$
\mathbb E[\widehat w]=w_*,
$$

并且

$$
\operatorname{Cov}(\widehat w)
=\sigma^2(X^\top X)^{-1}
=\frac{\sigma^2}{N}\widehat\Sigma^{-1}.
$$

所以，$X^\top X$ 的小特征值会产生较大的**参数方差**（Parameter Variance）。对于预测，使用**投影矩阵**（Projection Matrix）$H$：

$$
X\widehat w-Xw_*=H\varepsilon.
$$

由于 $H^2=H$、$H^\top=H$ 且 $\operatorname{tr}(H)=d$，

$$
\begin{aligned}
\mathbb E\left[\frac1N\|X\widehat w-Xw_*\|_2^2\right]
&=\frac1N\operatorname{tr}\left(H\,\mathbb E[\varepsilon\varepsilon^\top]\right)\\
&=\frac{\sigma^2d}{N}.
\end{aligned}
$$

因此

$$
\mathbb E[R_X(\widehat w)]-R_X^*=\frac{\sigma^2d}{N}.
$$

动画幻灯片证明的是较松的上界 $2\sigma^2\operatorname{tr}(H)/N$。在所给的同方差固定设计假设下，投影算子代数给出上述精确结果，并消除了因子 $2$。要使**超额风险**（Excess Risk）趋于零，结论仍要求 $d/N\to0$。

*资料来源：L4 幻灯片 19–25；Bach，命题 3.4–3.5。*


## 14. 秩亏 OLS（Rank-Deficient OLS）与 Moore–Penrose 伪逆（Moore–Penrose Pseudoinverse）

如果 $X^\top X$ 奇异，OLS 的预测仍然存在，但参数极小解可能不唯一。设**紧致奇异值分解**（Compact SVD）为

$$
X=U_r\Sigma_rV_r^\top,
\qquad
\Sigma_r=\operatorname{diag}(s_1,\ldots,s_r),
$$

其中 $r=\operatorname{rank}(X)$。Moore–Penrose 伪逆为

$$
X^+=V_r\Sigma_r^{-1}U_r^\top.
$$

**最小欧氏范数最小二乘解**（Minimum-Euclidean-Norm Least-Squares Solution）为

$$
\widehat w_{\min}=X^+y,
$$

而拟合向量是唯一的：

$$
X\widehat w_{\min}=XX^+y=H_ry,
\qquad
H_r=U_rU_r^\top.
$$

$H_r$ 是到 $\operatorname{col}(X)$ 上的投影算子，且 $\operatorname{tr}(H_r)=r$。因此，相同设计下精确的信号预测误差为

$$
\mathbb E\left[\frac1N\|X\widehat w-Xw_*\|_2^2\right]
=\frac{\sigma^2r}{N}.
$$

参数恢复则不同：$w_*$ 在 $\operatorname{null}(X)$ 中的分量无法识别。任意向量 $\widehat w_{\min}+v$，只要 $v\in\operatorname{null}(X)$，都会给出相同的训练预测。在秩亏问题中，仅说“OLS 解”是不完整的，除非同时说明**选择规则**（Selection Rule）。

*资料来源：L4 幻灯片 25 的练习；Bach，第 3.3 节与第 12.1.1 节。*


## 15. 训练误差（Training Error）、测试误差（Test Error）与乐观偏差（Optimism）

**训练残差**（Training Residual）为

$$
y-X\widehat w=(I-H)\varepsilon.
$$

由于 $I-H$ 是秩为 $N-d$ 的投影矩阵，

$$
\mathbb E\left[\frac1N\|y-X\widehat w\|_2^2\right]
=\sigma^2\left(1-\frac dN\right).
$$

相比之下，在相同设计处的独立响应 $y'$ 上，期望损失为

$$
\mathbb E\left[\frac1N\|y'-X\widehat w\|_2^2\right]
=\sigma^2+\frac{\sigma^2d}{N}.
$$

因此，训练损失的期望存在大小为

$$
\frac{2\sigma^2d}{N}.
$$

的**乐观偏差**（Optimism）。

这同时解释了三个事实：

- 即使增加的模型维数只是在拟合噪声，训练误差也会下降；
- 训练误差不是预测风险的无偏估计；
- **残差自由度校正**（Residual Degrees-of-Freedom Correction）使 $\|y-X\widehat w\|^2/(N-d)$ 成为 $\sigma^2$ 的无偏估计量。

这些等式适用于满秩固定设计。若秩为 $r$，则应把 $d$ 替换为 $r$。对于随机设计或模型误设，还会出现其他项。

*资料来源：Bach，第 3.5.1 节与习题 3.2。*


## 16. 为什么需要正则化（Regularization），以及岭回归（Ridge Regression）的推导

当 $d\ge N$、特征近似共线，或灵活的特征映射拟合了噪声时，OLS 会变得脆弱。岭回归加入**$\ell_2$ 惩罚**（L2 Penalty）。本笔记统一采用如下约定：

$$
\widehat w_\lambda
=\arg\min_w
\left\{
\frac1N\|y-Xw\|_2^2+\lambda\|w\|_2^2
\right\},
\qquad \lambda>0.
$$

求导并令梯度为零，得到

$$
\frac2N X^\top(Xw-y)+2\lambda w=0,
$$

因此

$$
\boxed{
\widehat w_\lambda=(X^\top X+N\lambda I)^{-1}X^\top y
=\frac1N(\widehat\Sigma+\lambda I)^{-1}X^\top y.
}
$$

对于任意非零 $z$，

$$
z^\top(X^\top X+N\lambda I)z
=\|Xz\|_2^2+N\lambda\|z\|_2^2>0,
$$

所以即使 $X^\top X$ 奇异，岭回归方程组仍有唯一解。

**尺度约定警告**（Scaling-Convention Warning）。 如果损失中不含 $1/N$，在相应重标度惩罚项后，解中出现的是 $\lambda I$ 而不是 $N\lambda I$。若在损失和惩罚中一致地加入 $1/2$，这些因子会抵消。在完整目标函数的约定明确之前，符号 $\lambda$ 的数值没有可比意义；这解释了幻灯片不同动画帧中看似不同的约定。

*资料来源：L4 幻灯片 26–29、33；Bach，第 3.6 节。*


## 17. 岭回归的谱收缩（Spectral Shrinkage）、对偶方法（Dual Method）与有效维数（Effective Dimension）

设 $X=U_r\Sigma_rV_r^\top$，其**奇异值**（Singular Values）$s_j>0$。则

$$
\widehat w_\lambda
=V_r\operatorname{diag}\left(\frac{s_j}{s_j^2+N\lambda}\right)U_r^\top y.
$$

岭回归的预测为

$$
\widehat y_\lambda=H_\lambda y,
\qquad
H_\lambda=X(X^\top X+N\lambda I)^{-1}X^\top,
$$

其**谱收缩因子**（Spectral Shrinkage Factor）为

$$
\frac{s_j^2}{s_j^2+N\lambda}
=\frac{\mu_j}{\mu_j+\lambda},
\qquad
\mu_j=\frac{s_j^2}{N}.
$$

具有较小 $s_j$、因而观测较弱的方向收缩最多。这使近奇异问题更加稳定。

恒等式

$$
(X^\top X+N\lambda I)^{-1}X^\top
=X^\top(XX^\top+N\lambda I)^{-1}
$$

给出**对偶形式**（Dual Form）

$$
\widehat w_\lambda=X^\top(XX^\top+N\lambda I)^{-1}y.
$$

求解一个 $N\times N$ 方程组可能比求解一个 $d\times d$ 方程组更便宜，尤其当 $d\gg N$ 时。

不同文献将下面两个相关量称为**有效自由度**（Effective Degrees of Freedom）：

$$
\operatorname{tr}(H_\lambda)
=\sum_j\frac{\mu_j}{\mu_j+\lambda},
$$

以及**方差自由度**（Variance Degrees of Freedom）

$$
\operatorname{tr}(H_\lambda^2)
=\sum_j\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

第二个量是 Bach 教材精确岭回归方差中出现的软参数计数。

*资料来源：Bach，第 3.6 节与习题 3.5。*


## 18. 一般偏差—方差分解（General Bias–Variance Decomposition）

令 $f^*(x)=\mathbb E[Y\mid X=x]$。训练集 $D$ 使学得的预测器 $\widehat f_D(x)$ 成为随机量。对于固定 $x$ 处的独立测试响应，

$$
Y_0=f^*(x)+\varepsilon_0,
\qquad
\mathbb E[\varepsilon_0\mid x]=0,
\qquad
\operatorname{Var}(\varepsilon_0\mid x)=\sigma^2(x).
$$

加上再减去 $\mathbb E_D[\widehat f_D(x)]$。测试噪声与训练数据相互独立，因此所有交叉项都消失：

$$
\boxed{
\mathbb E_{D,Y_0}\bigl[(Y_0-\widehat f_D(x))^2\mid x\bigr]
=\sigma^2(x)
+\bigl(f^*(x)-\mathbb E_D[\widehat f_D(x)]\bigr)^2
+\operatorname{Var}_D(\widehat f_D(x)).
}
$$

三项依次为**不可约噪声**（Irreducible Noise）、**偏差平方**（Squared Bias）和**方差**（Variance）。对 $X$ 平均该恒等式即可得到总体预测风险；将它应用于一组固定设计点，则得到幻灯片中的范数形式。

更复杂的模型往往具有更低的近似偏差，却有更高的**抽样方差**（Sampling Variance）。正则化有意增大偏差，同时希望以更大幅度降低方差。在设定正确的固定设计模型下，OLS 是无偏的，而 **Gauss–Markov 定理**（Gauss–Markov Theorem）保证它是方差最小的**线性无偏**（Linear Unbiased）估计量；但**有偏估计量**（Biased Estimator）仍可能具有更小的总测试 MSE。

*资料来源：L4 幻灯片 30–31、34；Bach，命题 3.3。*


## 19. 岭回归在固定设计下的精确偏差（Exact Bias）与方差（Variance）

使用 $\widehat\Sigma=X^\top X/N$ 和观测模型 $y=Xw_*+\varepsilon$。岭回归可写为

$$
\widehat w_\lambda
=(\widehat\Sigma+\lambda I)^{-1}
\left(\widehat\Sigma w_*+\frac1N X^\top\varepsilon\right).
$$

因此

$$
\mathbb E[\widehat w_\lambda]
=(\widehat\Sigma+\lambda I)^{-1}\widehat\Sigma w_*,
$$

**参数偏差**（Parameter Bias）为

$$
\operatorname{Bias}(\widehat w_\lambda)
=-\lambda(\widehat\Sigma+\lambda I)^{-1}w_*.
$$

只要 $\lambda>0$，岭回归通常就是有偏的。其**协方差**（Covariance）为

$$
\operatorname{Cov}(\widehat w_\lambda)
=\frac{\sigma^2}{N}
(\widehat\Sigma+\lambda I)^{-1}
\widehat\Sigma
(\widehat\Sigma+\lambda I)^{-1}.
$$

在**固定设计预测风险**（Fixed-Design Prediction Risk）中，精确的偏差平方项和方差项分别为

$$
B_\lambda
=\lambda^2w_*^\top
(\widehat\Sigma+\lambda I)^{-1}
\widehat\Sigma
(\widehat\Sigma+\lambda I)^{-1}w_*,
$$

$$
V_\lambda
=\frac{\sigma^2}{N}
\operatorname{tr}\left[
\widehat\Sigma^2(\widehat\Sigma+\lambda I)^{-2}
\right].
$$

若 $\widehat\Sigma v_j=\mu_jv_j$，则

$$
B_\lambda=\sum_j
\mu_j\left(\frac{\lambda}{\mu_j+\lambda}\right)^2(v_j^\top w_*)^2,
\qquad
V_\lambda=\frac{\sigma^2}{N}\sum_j
\left(\frac{\mu_j}{\mu_j+\lambda}\right)^2.
$$

随着 $\lambda$ 增大，方差减小，而偏差通常增大。预测意义下最优的 $\lambda$ 一般既不是零，也不是无穷大。

*资料来源：L4 幻灯片 32–34；Bach，命题 3.7。*


## 20. 选择正则化参数 $\lambda$（Regularization Parameter）、处理截距（Intercept）与最大后验（Maximum a Posteriori, MAP）解释

固定设计分析给出的一个理论上界为

$$
\mathbb E[R_X(\widehat w_\lambda)]-R_X^*
\le
\frac\lambda2\|w_*\|_2^2
+\frac{\sigma^2\operatorname{tr}(\widehat\Sigma)}{2\lambda N}.
$$

最小化该上界提示

$$
\lambda_*
=\frac{\sigma\sqrt{\operatorname{tr}(\widehat\Sigma)}}
{\|w_*\|_2\sqrt N},
$$

但 $w_*$ 和 $\sigma$ 未知，而且该上界未必紧。在实践中，应使用**验证集**（Validation Set）或**交叉验证**（Cross-Validation）选择 $\lambda$，并以实际测试时关心的指标进行评估。绝不能在最终**测试集**（Test Set）上调节 $\lambda$。

岭回归不具有**特征单位不变性**（Feature-Unit Invariance）：将某个特征乘以常数，会改变其系数受到惩罚的强度。应使用训练集统计量对非常数特征进行**标准化**（Standardization），然后再调节 $\lambda$；验证集和测试集必须应用同一个变换。

通常不惩罚截距。可以先中心化 $X$ 和 $y$，在无截距模型上拟合岭回归，再变换回原坐标；也可以使用如下**惩罚矩阵**（Penalty Matrix）：

$$
\lambda w^\top
\operatorname{diag}(0,1,\ldots,1)w.
$$

岭回归还有两种解释：

- **约束优化**（Constrained Optimization）：在 $\|w\|_2\le t$ 的约束下最小化训练损失；
- **贝叶斯最大后验**（Maximum a Posteriori, MAP）：高斯似然加上 $w$ 的零均值各向同性**高斯先验**（Gaussian Prior）。

先验方差与 $\lambda$ 之间的精确对应关系取决于损失的尺度约定。

*资料来源：L4 幻灯片 35–36；Bach，命题 3.8 与第 14.1 节。*


## 21. 一般正则化（General Regularization）、LASSO 与软阈值（Soft Thresholding）

正则化 ERM 的一般形式为

$$
\min_w\frac1N\sum_{i=1}^N
(y_i-\phi(x_i)^\top w)^2+\lambda\Omega(w).
$$

**正则化器**（Regularizer）表达了我们偏好哪类解：

- $\Omega(w)=\|w\|_2^2$：岭回归；实现平滑收缩并提高数值稳定性；
- $\Omega(w)=\|w\|_1$：**LASSO**（Least Absolute Shrinkage and Selection Operator）；促进系数稀疏；
- **群范数**（Group Norm）：选择或收缩预先定义的变量组；
- **核范数**（Nuclear Norm）：促进低秩矩阵；
- **原子范数**（Atomic Norm）：相对于给定原子集合促进结构化稀疏分解；
- 针对具体问题的平滑性、图结构或其他结构性惩罚。

应根据先验结构选择惩罚：$L_2$ 适合分散的小效应，$L_1$ 适合坐标稀疏，群范数适合块稀疏，核范数适合低秩结构，而当自然构件是特定于问题的原子时，应采用原子范数。

岭回归有闭式解；LASSO 不光滑，通常需要迭代算法。其基本机制可以从以下标量问题看出：

$$
\min_w\left\{\frac12w^2-xw+\lambda|w|\right\}.
$$

最优性条件 $0\in w-x+\lambda\partial|w|$ 给出**软阈值解**（Soft-Thresholding Solution）

$$
w^*=\operatorname{sign}(x)(|x|-\lambda)_+,
\qquad
(a)_+=\max(a,0).
$$

当 $|x|\le\lambda$ 时，取值会精确变为零，这解释了**稀疏性**（Sparsity）。相比之下，在正交坐标中，岭回归会把每个估计量乘以连续收缩因子，通常不会产生精确的零。

也可能出现**隐式正则化**（Implicit Regularization）：即使没有显式惩罚，**提前停止**（Early Stopping）、优化算法的解偏好、模型架构、**数据增强**（Data Augmentation）或特征构造也都可能限制模型的有效容量。

*资料来源：L4 幻灯片 36–37；解答手册，第 25 章习题 2 解答（第 65 页）。*


## 22. 实用的端到端流程（End-to-End Workflow）与常见误区（Common Pitfalls）

一个可靠的线性回归或岭回归流程如下：

1. 明确预测目标和测试时的数据分布。
2. 在拟合预处理步骤或选择**超参数**（Hyperparameter）之前划分数据。
3. 检查散点图、缺失值、离群点、**数据泄漏**（Data Leakage）和子群结构。
4. 只使用训练数据构造特征；若有合理依据，可加入非线性特征。
5. 使用岭回归或 LASSO 时，对特征进行中心化/标准化；通常不惩罚截距。
6. 使用 QR/SVD 或最小二乘例程拟合 OLS，不要显式求逆。
7. 通过验证集或交叉验证调节正则化强度。
8. 诊断残差模式和具有强影响力的观测。
9. 同时报告预测性能和评估设计；不要把关联表述为因果。

常见概念错误包括：

- 认为线性回归必须关于原始输入线性，而不是关于参数线性；
- 认为训练误差小就意味着**泛化误差**（Generalization Error）小；
- 把 $\sigma^2d/N$ 当作随机设计测试误差的精确公式；
- 声称 OLS 的无偏性要求高斯噪声；
- 在目标函数或特征尺度不同时直接比较 $\lambda$ 的数值；
- 无意中惩罚截距；
- 把一个不唯一的**插值解**（Interpolating Solution）当作唯一确定的解；
- 在缺乏因果设计时，对系数作**因果解释**（Causal Interpretation）。

*资料来源：对三份资料的综合整理。*


## 23. 进阶拓展：过参数化（Overparameterization）、隐式偏置（Implicit Bias）与双下降（Double Descent）

当 $d>N$ 且 $X$ 满行秩时，满足 $Xw=y$ 的**插值解**（Interpolating Solution）有无穷多个。**最小 $\ell_2$ 范数插值解**（Minimum-Norm Interpolator）为

$$
\widehat w_{\min}=X^+y=X^\top(XX^\top)^{-1}y.
$$

对未正则化平方损失执行**梯度下降**（Gradient Descent），若从 $w_0=0$ 初始化，则迭代始终位于 $\operatorname{row}(X)$ 中，并在收敛时收敛到这个最小范数解。这是算法与初始化的**隐式偏置**（Implicit Bias）；不同的 ERM 解选择规则可能选出不同的插值解。

以下精确的高斯/Wishart 公式将特征映射特化为 $\phi(x)=x$，其中 $x\sim\mathcal N(0,I_d)$：特征均值为零，设计矩阵中不含常数截距列。若要拟合截距，应先中心化数据并相应调整自由度。在这一各向同性高斯随机设计下，远离**插值阈值**（Interpolation Threshold）时，最小范数回归的**期望超额风险**（Expected Excess Risk）具有精确表达式：

$$
d\le N-2:
\qquad
\mathbb E[\text{excess risk}]
=\sigma^2\frac{d}{N-d-1},
$$

$$
d\ge N+2:
\qquad
\mathbb E[\text{excess risk}]
=\sigma^2\frac{N}{d-N-1}
+\|w_*\|_2^2\frac{d-N}{d}.
$$

在三个边界维数处，相应**逆 Wishart 矩阵**（Inverse-Wishart Matrix）的一阶矩不存在：

$$
\mathbb E[\text{excess risk}]=\infty,\qquad d\in\{N-1,N,N+1\}.
$$

因此，在这个理想化的期望中，风险具有无穷大的插值尖峰，随后可能再次下降，由此产生**双下降**（Double Descent）。这并不意味着增加参数总是有益。该现象取决于数据分布、信号、噪声、特征序列以及隐式或显式正则化。恰当调节的岭回归通常能够平滑插值峰值。

*资料来源：Bach，第 12.1–12.2 节。*


## 24. 进阶拓展：随机设计预测风险（Random-Design Prediction Risk）

在随机设计中，令 $\phi(X)$ 表示一个新的**总体特征向量**（Population Feature Vector），并定义

$$
\Sigma=\mathbb E[\phi(X)\phi(X)^\top],
\qquad
\widehat\Sigma=\frac1N X^\top X.
$$

在设定正确的同方差模型下，**总体超额风险**（Population Excess Risk）为

$$
R(w)-R(w_*)=\|w-w_*\|_\Sigma^2.
$$

给定一个满秩训练设计，OLS 的**条件协方差**（Conditional Covariance）为 $(\sigma^2/N)\widehat\Sigma^{-1}$，因此

$$
\mathbb E[R(\widehat w)-R(w_*)\mid X]
=\frac{\sigma^2}{N}
\operatorname{tr}(\Sigma\widehat\Sigma^{-1}).
$$

与固定设计不同，$\Sigma$ 和 $\widehat\Sigma$ 不会相消。对于精确的高斯/Wishart 公式，进一步假定 $\phi(x)=x\sim\mathcal N(0,\Sigma)$：特征均值为零，设计矩阵中不含常数截距列。若要拟合截距，应先中心化数据并相应调整自由度。当 $N>d+1$ 时，再对设计取平均可得

$$
\mathbb E[R(\widehat w)-R(w_*)]
=\sigma^2\frac{d}{N-d-1}.
$$

只有当 $d/N$ 很小时，它才近似等于 $\sigma^2d/N$。当 $d$ 从下方趋近 $N$ 时，该值发散。这说明课程中的固定设计迹计算只在其明确给定的范围内证明一致性；它本身不是一个关于未见输入泛化的完整**泛化定理**（Generalization Theorem）。

*资料来源：Bach，第 3.8 节。*


## 25. 幻灯片例题详解（Worked Exercises）

**A. 行列式问题（Determinant Problem）。** 令 $A(t)=\det(C+tD)$ 且 $M(t)=C+tD$。当 $M(t)$ 可逆时，**Jacobi 公式**（Jacobi's Formula）给出

$$
A'(t)=\det(C+tD)\operatorname{tr}\bigl((C+tD)^{-1}D\bigr).
$$

在奇异点同样有效的公式为

$$
A'(t)=\operatorname{tr}\bigl(\operatorname{adj}(C+tD)D\bigr).
$$

**B. 非线性特征（Nonlinear Features）。** 令设计矩阵满足 $\Phi_{i,:}=\phi(x_i)^\top$，则

$$
\min_w\|\Phi w-y\|_2^2
\quad\Longrightarrow\quad
\Phi^\top\Phi\widehat w=\Phi^\top y.
$$

若 $\Phi$ 满列秩，

$$
\widehat w=(\Phi^\top\Phi)^{-1}\Phi^\top y.
$$

**C. 奇异设计（Singular Design）。** 如果 $X^\top X$ 奇异，**最小范数 OLS 解**（Minimum-Norm OLS Solution）为 $\widehat w_{\min}=X^+y$。所有最小二乘解均为 $X^+y+v$，其中 $v\in\operatorname{null}(X)$；拟合投影算子为 $XX^+$。令 $r=\operatorname{rank}(X)$，则

$$
\mathbb E\left[\frac1N\|X\widehat w-Xw_*\|_2^2\right]
=\frac{\sigma^2r}{N}.
$$

**D. 岭回归的偏差与方差（Ridge Bias and Variance）。** 由第 19 节，

$$
\mathbb E[\widehat w_\lambda]-w_*
=-\lambda(\widehat\Sigma+\lambda I)^{-1}w_*,
$$

$$
\operatorname{Cov}(\widehat w_\lambda)
=\frac{\sigma^2}{N}
(\widehat\Sigma+\lambda I)^{-1}\widehat\Sigma
(\widehat\Sigma+\lambda I)^{-1}.
$$

这些推导补全了幻灯片中的提示，而不只是给出答案。

*资料来源：L4 课程管理部分的行列式问题，以及幻灯片 16、25、32–33。*


## 26. 最终检查清单（Final Checklist）与公式表（Formula Sheet）

学完本笔记后，你应当能够推导或解释以下每一项。

1. 平方损失 ERM，以及以中位数或分位数回归为目标的替代损失。
2. 一元回归斜率 $\widehat w=r s_y/s_x$ 与截距 $\bar y-\widehat w\bar x$。
3. 均值回归，以及 $\operatorname{RMSE}=s_y\sqrt{1-r^2}$。
4. 为什么条件均值最小化总体平方损失。
5. 为什么“关于参数线性”并不意味着“关于原始输入线性”。
6. OLS 正规方程及其投影解释。
7. 固定设计与随机设计的区别。
8. OLS 的无偏性、协方差与固定设计超额风险：

$$
\mathbb E[\widehat w]=w_*,
\qquad
\operatorname{Cov}(\widehat w)=\sigma^2(X^\top X)^{-1},
\qquad
\mathbb E[\text{excess risk}]=\frac{\sigma^2d}{N}.
$$

9. 训练损失的乐观偏差 $2\sigma^2d/N$，以及估计量 $\mathrm{RSS}/(N-d)$。
10. 参数解不唯一时的伪逆 OLS。
11. 采用本笔记尺度约定时的岭回归：

$$
\widehat w_\lambda=(X^\top X+N\lambda I)^{-1}X^\top y.
$$

12. 谱收缩、岭回归的对偶形式与有效自由度。
13. 岭回归精确的偏差平方项和方差项。
14. 为什么特征缩放、截距处理和验证很重要。
15. 岭回归、LASSO 与隐式正则化有何区别。
16. 为什么固定设计公式不能自动证明对未见输入的泛化。

**推荐复习顺序：** 在纸上重新推导第 3、6、8、13、16、19 节；然后不查看已展示的答案，独立完成第 25 节的四道练习。
